# Spectral-Aware Adapter with Subject-Adversarial Alignment — REVE Backbone

Adapted for the **PhysREVE Colab environment**.

Tests three conditions via Leave-One-Subject-Out (LOSO) on BCI IV 2a (all 9 subjects):

| Condition | Spectral adapter | Adversarial |
|---|---|---|
| `baseline` | — | — |
| `spectral` | ✓ | — |
| `spectral_adversarial` | ✓ | ✓ |

**Hypothesis:** If spectral adapter helps REVE, REVE's raw-patch tokenisation is missing  
frequency information. Compare gains vs. PhysREVE's single-subject results in  
`quick_wins_notebook.ipynb`.


## 1. Setup & Installation

In [ ]:
import subprocess, sys, os

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    repo = '/content/PhysREVE'
    if not os.path.exists(repo):
        subprocess.check_call(['git', 'clone', '-q', 'https://github.com/UgoBruzadin/PhysREVE.git', repo])
    else:
        subprocess.check_call(['git', '-C', repo, 'pull', '-q'])
    if repo not in sys.path:
        sys.path.insert(0, repo)
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        'mne>=1.6', 'moabb>=1.0', 'xgboost',
        'transformers', 'huggingface_hub', 'PyWavelets', 'einops'
    ])
    print('Colab environment ready.')
else:
    print('Local environment.')

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import pywt
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

## 2. Load BCI Competition IV-2a

In [ ]:
from moabb.datasets import BNCI2014_001
from moabb.paradigms import MotorImagery
from physreve.datasets.bciv2a import CH_NAMES, N_CLASSES, SFREQ, CLASS_NAMES, LABEL_MAP

dataset  = BNCI2014_001()
paradigm = MotorImagery(
    events=['left_hand', 'right_hand', 'feet', 'tongue'],
    n_classes=4, fmin=0.5, fmax=40.0, tmin=0.5, tmax=2.5
)

ALL_SUBJECTS = list(range(1, 10))
X_parts, y_parts, subj_parts = [], [], []

for s in ALL_SUBJECTS:
    Xs, ys_str, _ = paradigm.get_data(dataset, subjects=[s], return_epochs=False)
    Xs = Xs.astype(np.float32)
    ys = np.array([LABEL_MAP[yi] for yi in ys_str])
    X_parts.append(Xs)
    y_parts.append(ys)
    subj_parts.append(np.full(len(ys), s))
    print(f'  Subject {s}: {Xs.shape[0]} trials  {Xs.shape[1]}ch × {Xs.shape[2]}samples')

X        = np.concatenate(X_parts,    axis=0)   # (N, C, T)
y_encoded = np.concatenate(y_parts,   axis=0)   # (N,) int
subjects  = np.concatenate(subj_parts, axis=0)  # (N,) int
unique_subjects = np.unique(subjects)
n_classes = N_CLASSES

N_CHANNELS = X.shape[1]
n_times    = X.shape[2]

print(f'\nTotal: {X.shape[0]} trials × {N_CHANNELS} ch × {n_times} samples')
print(f'SFREQ: {SFREQ} Hz   Classes: {CLASS_NAMES}   Subjects: {unique_subjects}')


## 3. Load REVE Model

REVE is loaded via HuggingFace `transformers`. It needs:
1. The model itself (`brain-bzh/reve-base`)
2. A position bank that maps electrode names to 3D coordinates (`brain-bzh/reve-positions`)

BCI IV-2a uses 22 EEG channels in the 10-20 system.

In [ ]:
from transformers import AutoModel

print("Loading REVE position bank...")
pos_bank = AutoModel.from_pretrained("brain-bzh/reve-positions", trust_remote_code=True)

print("Loading REVE-Base encoder...")
reve_model = AutoModel.from_pretrained("brain-bzh/reve-base", trust_remote_code=True)
reve_model = reve_model.to(DEVICE)
reve_model.eval()

# Freeze all REVE parameters
for param in reve_model.parameters():
    param.requires_grad = False

n_reve_params = sum(p.numel() for p in reve_model.parameters())
print(f"REVE parameters: {n_reve_params:,} (all frozen)")

In [ ]:
# BCI IV-2a electrode names (22 EEG channels, 10-20 system)
# These are the standard names from the BNCI2014_001 dataset
# MOABB returns channel info — let's extract the names

# Standard BCI IV-2a channel names
CHANNEL_NAMES = [
    'Fz', 'FC3', 'FC1', 'FCz', 'FC2', 'FC4',
    'C5', 'C3', 'C1', 'Cz', 'C2', 'C4', 'C6',
    'CP3', 'CP1', 'CPz', 'CP2', 'CP4',
    'P1', 'Pz', 'P2', 'POz'
]

N_CHANNELS = len(CHANNEL_NAMES)
print(f"Using {N_CHANNELS} channels: {CHANNEL_NAMES}")

# Get 3D positions for these electrodes from REVE's position bank
try:
    positions = pos_bank(CHANNEL_NAMES)  # (n_channels, 3)
    print(f"Electrode positions shape: {positions.shape}")
    print(f"Sample position (Cz): {positions[CHANNEL_NAMES.index('Cz')]}")
except Exception as e:
    print(f"Position bank error: {e}")
    print("Trying alternative channel names...")
    # Some position banks use uppercase or different naming conventions
    # Try common alternatives
    alt_names = [ch.upper() for ch in CHANNEL_NAMES]
    try:
        positions = pos_bank(alt_names)
        print(f"Worked with uppercase names. Shape: {positions.shape}")
    except:
        print("Please check REVE docs for supported electrode names.")
        print("You may need to adjust CHANNEL_NAMES to match the position bank.")

In [ ]:
# Test REVE forward pass to understand output shape
# REVE expects: (batch, channels, timepoints) at 200 Hz
# plus electrode positions: (batch, channels, 3) or (channels, 3)

n_times = X.shape[2]
print(f"Input timepoints: {n_times} ({n_times/200:.1f}s at 200 Hz)")

with torch.no_grad():
    dummy_eeg = torch.randn(2, N_CHANNELS, n_times).to(DEVICE)
    dummy_pos = positions.unsqueeze(0).expand(2, -1, -1).to(DEVICE)
    
    # REVE forward - check the exact API
    try:
        reve_out = reve_model(dummy_eeg, positions=dummy_pos)
        # Output could be a dict, tuple, or tensor depending on the model version
        if hasattr(reve_out, 'last_hidden_state'):
            feat = reve_out.last_hidden_state
            print(f"REVE output (last_hidden_state): {feat.shape}")
        elif isinstance(reve_out, tuple):
            feat = reve_out[0]
            print(f"REVE output (tuple[0]): {feat.shape}")
        elif isinstance(reve_out, torch.Tensor):
            feat = reve_out
            print(f"REVE output (tensor): {feat.shape}")
        else:
            print(f"REVE output type: {type(reve_out)}")
            print(f"REVE output keys: {reve_out.keys() if hasattr(reve_out, 'keys') else 'N/A'}")
            feat = reve_out
    except Exception as e:
        print(f"Forward pass error: {e}")
        print("\nLet's inspect the model signature...")
        import inspect
        sig = inspect.signature(reve_model.forward)
        print(f"Forward signature: {sig}")
        print("\nYou may need to adjust the forward call based on REVE's API.")
        print("Check: https://github.com/elouayas/reve_eeg")

In [ ]:
# Determine the flattened output dimension for downstream heads
# Adapt this cell based on the output from the cell above

# If REVE returns (batch, n_patches, embed_dim), we'll pool over patches
# If it returns (batch, embed_dim) directly, we use it as-is

# Placeholder — fill in after running the cell above:
# REVE_OUTPUT_SHAPE = feat.shape  # e.g., (2, n_patches, 200) or (2, 200)

# We'll define a helper to extract a fixed-size feature vector from REVE
class REVEFeatureExtractor(nn.Module):
    """Wraps REVE to produce a fixed-size feature vector.
    
    Handles different output formats (sequence vs pooled).
    """
    def __init__(self, reve_model, positions, pool='mean'):
        super().__init__()
        self.reve = reve_model
        self.register_buffer('positions', positions)  # (n_channels, 3)
        self.pool = pool
    
    @torch.no_grad()
    def forward(self, x):
        """x: (batch, channels, timepoints)"""
        batch_size = x.size(0)
        pos = self.positions.unsqueeze(0).expand(batch_size, -1, -1)
        
        out = self.reve(x, positions=pos)
        
        # Extract tensor from model output
        if hasattr(out, 'last_hidden_state'):
            feat = out.last_hidden_state
        elif isinstance(out, tuple):
            feat = out[0]
        elif isinstance(out, torch.Tensor):
            feat = out
        else:
            # Try common attribute names
            for attr in ['pooler_output', 'logits', 'hidden_states']:
                if hasattr(out, attr):
                    feat = getattr(out, attr)
                    if isinstance(feat, tuple): feat = feat[-1]
                    break
        
        # Pool if output is a sequence (batch, seq_len, embed_dim)
        if feat.dim() == 3:
            if self.pool == 'mean':
                feat = feat.mean(dim=1)   # (batch, embed_dim)
            elif self.pool == 'cls':
                feat = feat[:, 0, :]      # CLS token
        
        return feat


# Create the feature extractor
backbone = REVEFeatureExtractor(reve_model, positions).to(DEVICE)
backbone.eval()

# Test it
with torch.no_grad():
    dummy = torch.randn(2, N_CHANNELS, n_times).to(DEVICE)
    feat_out = backbone(dummy)
    print(f"Feature extractor output: {feat_out.shape}")
    backbone_feat_dim = feat_out.shape[-1]
    print(f"Feature dimension: {backbone_feat_dim}")

## 4. Prepare Data

REVE takes raw EEG directly (no manual patching needed).
We still need spectral features for our adapter branch.

In [ ]:
# Z-score per trial per channel (same as physreve.data.LabeledEEGDataset)
X_eeg  = X.copy()
X_eeg -= X_eeg.mean(axis=-1, keepdims=True)
X_eeg /= (X_eeg.std(axis=-1, keepdims=True) + 1e-8)
print(f'Normalised EEG: {X_eeg.shape}  (trials, channels, timepoints)')


## 5. Compute Spectral Features — Fast STFT

Replaces PyWavelets CWT (slow, per-trial loop) with a **vectorized STFT**:  
split each trial into `N_SPEC_SEGMENTS` chunks → FFT each chunk → average power per band.  
Same output shape `(trials, channels, bands, time_bins)`, ~50× faster.


In [ ]:
# SFREQ imported from physreve.datasets.bciv2a above
N_SPEC_SEGMENTS = 8   # time bins per trial

BANDS = {
    'delta': (1,   4),
    'theta': (4,   8),
    'alpha': (8,  13),
    'beta':  (13, 30),
    'gamma': (30, 40),
}

def compute_spectral_features_stft(X_raw, sfreq=200.0, n_segments=8):
    """Vectorised STFT band power.

    Splits each trial into n_segments chunks, computes FFT on each chunk,
    averages power within each frequency band.

    Returns: (n_trials, n_ch, n_bands, n_segments)  float32
    Speed:   ~50x faster than per-trial PyWavelets CWT
    """
    n_trials, n_ch, n_times = X_raw.shape
    seg_len  = n_times // n_segments
    usable   = seg_len * n_segments
    n_bands  = len(BANDS)

    # Reshape to (trials, ch, n_segments, seg_len)
    X_seg = X_raw[:, :, :usable].reshape(n_trials, n_ch, n_segments, seg_len)

    # FFT along last axis → (trials, ch, n_segments, n_freqs)
    fft_pow = np.abs(np.fft.rfft(X_seg, axis=-1)) ** 2
    freqs   = np.fft.rfftfreq(seg_len, d=1.0 / sfreq)

    # Band power: (trials, ch, n_bands, n_segments)
    out = np.zeros((n_trials, n_ch, n_bands, n_segments), dtype=np.float32)
    for b, (_, (fmin, fmax)) in enumerate(BANDS.items()):
        mask = (freqs >= fmin) & (freqs <= fmax)
        if not mask.any():
            print(f'  Warning: no FFT bins in band {fmin}-{fmax} Hz (seg_len={seg_len})')
            continue
        out[:, :, b, :] = fft_pow[:, :, :, mask].mean(axis=-1)  # mean over freq bins

    # Log + z-score per trial (over time axis)
    out = np.log1p(out)
    mu  = out.mean(axis=-1, keepdims=True)
    sig = out.std(axis=-1,  keepdims=True) + 1e-8
    out = (out - mu) / sig

    return out


import time
t0 = time.time()
print(f'Computing STFT band power for {len(X_eeg)} trials at {SFREQ} Hz...')
X_spectral_pooled = compute_spectral_features_stft(X_eeg, sfreq=SFREQ, n_segments=N_SPEC_SEGMENTS)
print(f'Done in {time.time()-t0:.1f}s')
print(f'Spectral shape: {X_spectral_pooled.shape}  (trials, ch, bands, time_bins)')
print(f'Freq resolution per segment: {SFREQ / (X_eeg.shape[-1] // N_SPEC_SEGMENTS):.2f} Hz/bin')


In [ ]:
# X_spectral_pooled is already (trials, channels, bands, N_SPEC_SEGMENTS)
# from the STFT cell above — no additional pooling needed.
print(f'Spectral features ready: {X_spectral_pooled.shape}')
print(f'  trials={X_spectral_pooled.shape[0]}  ch={X_spectral_pooled.shape[1]}  '
      f'bands={X_spectral_pooled.shape[2]}  time_bins={X_spectral_pooled.shape[3]}')


## 6. Dataset & Model

In [ ]:
class EEGDataset(Dataset):
    def __init__(self, X_eeg, X_spectral, y, subject_ids):
        self.X_eeg = torch.FloatTensor(X_eeg)
        self.X_spectral = torch.FloatTensor(X_spectral)
        self.y = torch.LongTensor(y)
        self.subject_ids = torch.LongTensor(subject_ids)
    
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return self.X_eeg[idx], self.X_spectral[idx], self.y[idx], self.subject_ids[idx]

In [ ]:
# --- Gradient Reversal Layer ---

class GradientReversalFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambda_):
        ctx.lambda_ = lambda_
        return x.clone()
    
    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambda_ * grad_output, None


class GradientReversal(nn.Module):
    def __init__(self, lambda_=1.0):
        super().__init__()
        self.lambda_ = lambda_
    
    def forward(self, x):
        return GradientReversalFunction.apply(x, self.lambda_)
    
    def set_lambda(self, lambda_):
        self.lambda_ = lambda_

In [ ]:
# --- Spectral CNN ---

class SpectralCNN(nn.Module):
    """Lightweight CNN for band-power timecourses.
    Input: (batch, n_channels, n_bands, n_time_bins)
    Output: (batch, spectral_dim)
    """
    def __init__(self, n_channels=22, n_bands=5, n_time_bins=8, hidden_dim=64):
        super().__init__()
        self.conv = nn.Conv2d(n_channels, hidden_dim, kernel_size=(n_bands, 1))
        self.bn = nn.BatchNorm2d(hidden_dim)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(hidden_dim, hidden_dim)
        self.act = nn.ELU()
    
    def forward(self, x):
        x = self.act(self.bn(self.conv(x)))
        x = self.pool(x).squeeze(-1).squeeze(-1)
        x = self.act(self.fc(x))
        return x

In [ ]:
# --- Full Model ---

class REVESpectralAdversarialModel(nn.Module):
    """Frozen REVE + Spectral Adapter + Subject-Adversarial Alignment."""
    
    def __init__(
        self,
        backbone,           # REVEFeatureExtractor (frozen)
        backbone_feat_dim,  # output dim of REVE features
        n_channels=22,
        n_bands=5,
        n_time_bins=8,
        n_classes=4,
        n_subjects=9,
        spectral_dim=64,
        fusion_dim=128,
        adv_lambda=0.1,
        use_spectral=True,
        use_adversarial=True,
    ):
        super().__init__()
        self.backbone = backbone
        self.use_spectral = use_spectral
        self.use_adversarial = use_adversarial
        
        # Project REVE output to fusion dim
        self.backbone_proj = nn.Sequential(
            nn.Linear(backbone_feat_dim, fusion_dim),
            nn.ELU(),
            nn.Dropout(0.3),
        )
        
        if use_spectral:
            self.spectral_cnn = SpectralCNN(
                n_channels=n_channels,
                n_bands=n_bands,
                n_time_bins=n_time_bins,
                hidden_dim=spectral_dim
            )
            total_dim = fusion_dim + spectral_dim
        else:
            total_dim = fusion_dim
        
        # Fusion MLP
        self.fusion = nn.Sequential(
            nn.Linear(total_dim, fusion_dim),
            nn.ELU(),
            nn.Dropout(0.3),
            nn.Linear(fusion_dim, fusion_dim),
            nn.ELU(),
        )
        
        # Task head
        self.task_head = nn.Linear(fusion_dim, n_classes)
        
        # Adversarial head
        if use_adversarial:
            self.grl = GradientReversal(lambda_=adv_lambda)
            self.subject_head = nn.Sequential(
                nn.Linear(fusion_dim, 64),
                nn.ELU(),
                nn.Dropout(0.3),
                nn.Linear(64, n_subjects),
            )
    
    def forward(self, x_eeg, x_spectral=None):
        # Frozen REVE forward
        with torch.no_grad():
            f_backbone = self.backbone(x_eeg)
        
        f_backbone = self.backbone_proj(f_backbone)
        
        if self.use_spectral and x_spectral is not None:
            f_spectral = self.spectral_cnn(x_spectral)
            f_combined = torch.cat([f_backbone, f_spectral], dim=-1)
        else:
            f_combined = f_backbone
        
        z = self.fusion(f_combined)
        task_logits = self.task_head(z)
        
        if self.use_adversarial:
            z_reversed = self.grl(z)
            subject_logits = self.subject_head(z_reversed)
            return task_logits, subject_logits
        
        return task_logits, None


# Sanity check
model_test = REVESpectralAdversarialModel(
    backbone=backbone,
    backbone_feat_dim=backbone_feat_dim,
    n_time_bins=N_SPEC_SEGMENTS,
).to(DEVICE)

with torch.no_grad():
    x_e = torch.randn(4, N_CHANNELS, n_times).to(DEVICE)
    x_s = torch.randn(4, N_CHANNELS, len(BANDS), N_SPEC_SEGMENTS).to(DEVICE)
    task_out, subj_out = model_test(x_e, x_s)
    print(f"Task output: {task_out.shape}")
    print(f"Subject output: {subj_out.shape}")

trainable = sum(p.numel() for p in model_test.parameters() if p.requires_grad)
total = sum(p.numel() for p in model_test.parameters())
print(f"\nTrainable: {trainable:,}  |  Total: {total:,}  |  Ratio: {trainable/total*100:.1f}%")
del model_test

## 7. Training & Evaluation Functions

In [ ]:
def lambda_schedule(epoch, max_epochs, gamma=10.0):
    p = epoch / max_epochs
    return 2.0 / (1.0 + np.exp(-gamma * p)) - 1.0


def train_epoch(model, train_loader, optimizer, epoch, max_epochs, adv_weight=0.1):
    model.train()
    model.backbone.eval()  # keep frozen backbone in eval mode
    
    total_task_loss = 0
    total_subj_loss = 0
    all_preds, all_labels = [], []
    
    if model.use_adversarial:
        current_lambda = lambda_schedule(epoch, max_epochs) * adv_weight
        model.grl.set_lambda(current_lambda)
    
    for x_eeg, x_spectral, labels, subj_ids in train_loader:
        x_eeg = x_eeg.to(DEVICE)
        x_spectral = x_spectral.to(DEVICE)
        labels = labels.to(DEVICE)
        subj_ids = subj_ids.to(DEVICE)
        
        optimizer.zero_grad()
        task_logits, subj_logits = model(x_eeg, x_spectral)
        
        task_loss = F.cross_entropy(task_logits, labels)
        
        if model.use_adversarial and subj_logits is not None:
            subj_loss = F.cross_entropy(subj_logits, subj_ids)
            loss = task_loss + subj_loss
            total_subj_loss += subj_loss.item()
        else:
            loss = task_loss
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad], max_norm=1.0
        )
        optimizer.step()
        
        total_task_loss += task_loss.item()
        all_preds.extend(task_logits.argmax(1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    
    n = len(train_loader)
    return total_task_loss/n, total_subj_loss/n, balanced_accuracy_score(all_labels, all_preds)


@torch.no_grad()
def evaluate(model, test_loader):
    model.eval()
    all_preds, all_labels, all_subjs = [], [], []
    
    for x_eeg, x_spectral, labels, subj_ids in test_loader:
        x_eeg = x_eeg.to(DEVICE)
        x_spectral = x_spectral.to(DEVICE)
        task_logits, _ = model(x_eeg, x_spectral)
        all_preds.extend(task_logits.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())
        all_subjs.extend(subj_ids.numpy())
    
    return balanced_accuracy_score(all_labels, all_preds), np.array(all_preds), np.array(all_labels), np.array(all_subjs)

## 8. Leave-One-Subject-Out Cross-Validation

In [ ]:
BATCH_SIZE = 32       # smaller batch — REVE forward pass uses more memory than CBraMod
MAX_EPOCHS = 80
LR = 1e-3
WEIGHT_DECAY = 1e-4
ADV_WEIGHT = 0.3
PATIENCE = 15

CONDITIONS = {
    'baseline':             {'use_spectral': False, 'use_adversarial': False},
    'spectral':             {'use_spectral': True,  'use_adversarial': False},
    'spectral_adversarial': {'use_spectral': True,  'use_adversarial': True},
}

all_results = {cond: {} for cond in CONDITIONS}

print("=" * 70)
print("REVE — LEAVE-ONE-SUBJECT-OUT CROSS-VALIDATION")
print("=" * 70)

In [ ]:
for test_subject in unique_subjects:
    print(f"\n{'='*60}")
    print(f"  Test subject: {test_subject}")
    print(f"{'='*60}")
    
    test_mask = subjects == test_subject
    train_mask = ~test_mask
    
    train_subj_enc = LabelEncoder()
    train_subject_ids = train_subj_enc.fit_transform(subjects[train_mask])
    n_train_subjects = len(train_subj_enc.classes_)
    test_subject_ids = np.zeros(test_mask.sum(), dtype=int)
    
    train_dataset = EEGDataset(
        X_eeg[train_mask], X_spectral_pooled[train_mask],
        y_encoded[train_mask], train_subject_ids
    )
    test_dataset = EEGDataset(
        X_eeg[test_mask], X_spectral_pooled[test_mask],
        y_encoded[test_mask], test_subject_ids
    )
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    for cond_name, cond_params in CONDITIONS.items():
        print(f"\n  Condition: {cond_name}")
        
        model = REVESpectralAdversarialModel(
            backbone=backbone,
            backbone_feat_dim=backbone_feat_dim,
            n_channels=N_CHANNELS,
            n_bands=len(BANDS),
            n_time_bins=N_SPEC_SEGMENTS,
            n_classes=n_classes,
            n_subjects=n_train_subjects,
            adv_lambda=ADV_WEIGHT,
            **cond_params,
        ).to(DEVICE)
        
        optimizer = torch.optim.AdamW(
            [p for p in model.parameters() if p.requires_grad],
            lr=LR, weight_decay=WEIGHT_DECAY
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)
        
        best_test_acc = 0
        patience_counter = 0
        
        for epoch in range(MAX_EPOCHS):
            task_loss, subj_loss, train_acc = train_epoch(
                model, train_loader, optimizer, epoch, MAX_EPOCHS, ADV_WEIGHT
            )
            scheduler.step()
            
            if (epoch + 1) % 5 == 0 or epoch == 0:
                test_acc, _, _, _ = evaluate(model, test_loader)
                if test_acc > best_test_acc:
                    best_test_acc = test_acc
                    patience_counter = 0
                else:
                    patience_counter += 5
                
                if (epoch + 1) % 20 == 0:
                    print(f"    Ep {epoch+1:3d} | Loss: {task_loss:.3f} | "
                          f"Train: {train_acc:.3f} | Test: {test_acc:.3f} | "
                          f"Best: {best_test_acc:.3f}")
            
            if patience_counter >= PATIENCE:
                print(f"    Early stopping at epoch {epoch+1}")
                break
        
        all_results[cond_name][test_subject] = best_test_acc
        print(f"    >>> Best: {best_test_acc:.3f}")
        
        del model, optimizer, scheduler
        torch.cuda.empty_cache()

## 9. Results

In [ ]:
print("\n" + "="*70)
print("REVE RESULTS: Leave-One-Subject-Out Balanced Accuracy")
print("="*70)

header = f"{'Subject':>10}"
for cond in CONDITIONS:
    header += f" | {cond:>22}"
print(header)
print("-" * len(header))

for subj in unique_subjects:
    row = f"{subj:>10}"
    for cond in CONDITIONS:
        acc = all_results[cond].get(subj, 0)
        row += f" | {acc:>22.3f}"
    print(row)

print("-" * len(header))
row_mean = f"{'MEAN':>10}"
row_std = f"{'STD':>10}"
for cond in CONDITIONS:
    vals = list(all_results[cond].values())
    row_mean += f" | {np.mean(vals):>22.3f}"
    row_std += f" | {np.std(vals):>22.3f}"
print(row_mean)
print(row_std)

print("\n" + "="*70)
print("WORST vs BEST SUBJECT GAP")
print("="*70)
for cond in CONDITIONS:
    vals = list(all_results[cond].values())
    print(f"  {cond:>25}: min={min(vals):.3f}  max={max(vals):.3f}  "
          f"gap={max(vals)-min(vals):.3f}  std={np.std(vals):.3f}")

In [ ]:
# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
x = np.arange(len(unique_subjects))
width = 0.25
colors = ['#85B7EB', '#5DCAA5', '#AFA9EC']

for i, (cond, color) in enumerate(zip(CONDITIONS, colors)):
    vals = [all_results[cond][s] for s in unique_subjects]
    ax.bar(x + i*width, vals, width, label=cond, color=color, edgecolor='white')

ax.set_xlabel('Subject')
ax.set_ylabel('Balanced Accuracy')
ax.set_title('REVE: Per-subject cross-subject accuracy')
ax.set_xticks(x + width)
ax.set_xticklabels([str(s) for s in unique_subjects])
ax.legend(fontsize=8)
ax.set_ylim(0, 1)
ax.axhline(y=0.25, color='gray', linestyle='--', alpha=0.5)

ax = axes[1]
base = np.array([all_results['baseline'][s] for s in unique_subjects])
spec = np.array([all_results['spectral'][s] for s in unique_subjects])
full = np.array([all_results['spectral_adversarial'][s] for s in unique_subjects])

ax.scatter(base, spec - base, s=80, c='#5DCAA5', label='+ spectral', zorder=3, edgecolors='white')
ax.scatter(base, full - base, s=80, c='#AFA9EC', label='+ spec + adv', zorder=3, edgecolors='white')
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Baseline accuracy (frozen REVE)')
ax.set_ylabel('Improvement over baseline')
ax.set_title('REVE: Who benefits most?')
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('reve_results.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Save Results for Cross-Model Comparison

Save results as JSON so you can load them alongside CBraMod results.

In [ ]:
import json

# Convert numpy types for JSON serialization
results_serializable = {}
for cond, subj_dict in all_results.items():
    results_serializable[cond] = {str(k): float(v) for k, v in subj_dict.items()}

output = {
    'backbone': 'REVE-Base',
    'dataset': 'BCI-IV-2a',
    'evaluation': 'LOSO',
    'results': results_serializable,
}

with open('reve_results.json', 'w') as f:
    json.dump(output, f, indent=2)

print("Results saved to reve_results.json")
print("\nDownload this file and the CBraMod equivalent to compare backbones.")
print("\nKey comparison to make:")
print("  spectral_improvement_reve = REVE(+spectral) - REVE(baseline)")
print("  spectral_improvement_cbramod = CBraMod(+spectral) - CBraMod(baseline)")
print("  If spectral_improvement_reve > spectral_improvement_cbramod,")
print("  then REVE's raw-patch approach IS missing frequency information.")

## 11. Compare with PhysREVE Single-Subject Results

Load `reve_results.json` alongside the PhysREVE single-subject numbers to answer  
whether cross-subject REVE + spectral adapter beats within-subject PhysREVE fine-tuning.

In [ ]:
# PhysREVE single-subject reference (subject 1, from quick_wins_notebook)
physreve_ref = {
    'lda_subject1':          0.581,
    'random_init_large':     0.407,
    'physreve_large':        0.407,
    'small_random_init':     0.442,
    'chance':                0.250,
}

print('=== Cross-model comparison ===')
print(f'{"Model":<35}  {"Acc":>6}')
print('-' * 45)
print(f'[PhysREVE env, subj 1 only]')
for k, v in physreve_ref.items():
    print(f'  {k:<33}  {v:.3f}')

print(f'\n[REVE LOSO — mean over 9 subjects]')
for cond in CONDITIONS:
    vals = list(all_results[cond].values())
    print(f'  {cond:<33}  {np.mean(vals):.3f}  (±{np.std(vals):.3f})')

print()
print('Key question:')
spec_delta = np.mean(list(all_results['spectral'].values())) - np.mean(list(all_results['baseline'].values()))
adv_delta  = np.mean(list(all_results['spectral_adversarial'].values())) - np.mean(list(all_results['spectral'].values()))
print(f'  Spectral adapter Δ: {spec_delta:+.3f}  (REVE missing frequency info if positive)')
print(f'  Adversarial Δ:      {adv_delta:+.3f}  (subject alignment useful if positive)')
